# MeChess 6/7: test sets for the human-likeness study (CPU notebook)

**Accelerator: None.** An adapter around `chessme humanlike-testset` and `chessme humanlike-allie`.

1. **Our test set (T-B):** positions from 24 byte windows spread over one month of the Lichess database (a month **after** Maia-3's training data ends), blitz only, bots excluded, 4 positions per game chosen by a hash of the game id, with the clocks, the history and the rating of both players.
   Positions are thinned to at most 60,000 per 100-point rating bin so the extremes are represented. About 100 games per second per core are parsed, so this takes roughly two hours on four cores.
2. **The Allie test set (T-A):** the public 18,239-game test set of Allie / Maia-3, filtered exactly as the papers do (884,049 positions).

Output: `testset_TB.jsonl.gz`, `testset_TA.jsonl.gz` and their manifests (with content hashes). Notebook 7 scores models on them.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here (not needed when the repository is attached as a dataset)
MONTH = "2026-06"                                       # a month on https://database.lichess.org, later than July 2025 (Maia-3's training data ends then)
PARTS = 700
WINDOWS = "14,43,72,101,130,159,188,217,246,275,304,333,362,391,420,449,478,507,536,565,594,623,652,681"
WORKERS = 4
ALLIE = True                                            # False: skip the Allie test set (a second, independent test set of ours needs only T-B)
CAP_PER_BIN = 60000
OUT = "/kaggle/working/humanlike" if os.path.exists("/kaggle") else "humanlike_local"
SOURCE = f"https://database.lichess.org/standard/lichess_db_standard_rated_{MONTH}.pgn.zst"
ALLIE_URL = "https://huggingface.co/datasets/yimingzhang/allie-data/resolve/main/lichess-2022-blitz-test/2022-test-annotated.jsonl"

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    attached = glob.glob("/kaggle/input/**/chessme/__init__.py", recursive=True)     # the repository attached as a dataset
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), "MeChess")
    else:                                                           # Kaggle: fetch the repository (Internet must be on)
        if "<you>" in REPO_URL:
            raise SystemExit("Attach the repository as a dataset, or set REPO_URL.")
        sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "chess==1.11.2", "numpy", "pyyaml", "requests", "zstandard", "huggingface-hub")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

## Step 1. Our test set (parallel, balanced over rating)

In [ ]:
sh(*CLI, "humanlike-testset", "--source", SOURCE, "--parts", str(PARTS), "--windows", WINDOWS, "--workers", str(WORKERS),
   "--cap-per-bin", str(CAP_PER_BIN), "--per-game", "4", "--shard-dir", "/tmp/shards", "--out", f"{OUT}/testset_TB.jsonl.gz")

## Step 2. The Allie test set, filtered as in the papers

In [ ]:
os.makedirs(OUT, exist_ok=True)
if ALLIE:
    sh("curl", "-fsSL", "-o", "/tmp/allie_test.jsonl", ALLIE_URL)
    sh(*CLI, "humanlike-allie", "--file", "/tmp/allie_test.jsonl", "--out", f"{OUT}/testset_TA.jsonl.gz")